In [1]:
import fitness_landscape as fl
import os
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path
import networkx as nx
from tqdm import tqdm

/home/matthewspence/graph-ruggedness-de/.env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-23 19:42:38,724	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [27]:


# Load folding landscape
df_folding = pd.read_csv('../data_files/combinatorial_core/FYN_folding_fitness.csv')

# Load binding landscape
df_binding = pd.read_csv('../data_files/combinatorial_core/FYN_binding_fitness.csv')

# Inner join to keep only sequences found in both assays
df_intersect = pd.merge(
    df_binding, 
    df_folding, 
    on='sequence', 
    how='inner', 
    suffixes=('_bind', '_fold')
)

df_intersect.dropna(inplace=True)

sequences = [fl.BaseNumpySequence(sequence) for sequence in df_intersect['sequence']]
fintess_binding = np.array(df_intersect['fitness_bind'])
fitness_folding = np.array(df_intersect['fitness_fold'])

# Scale K to sqrt of number of sequences
knn_k = max(int(np.sqrt(len(sequences))), 2)

# Construct fitness landscape
landscape = fl.FitnessLandscape.build(
    sequences,
    graph="knn",
    k=knn_k,
    backend="auto",
    _compute_hamming_edges=False
)

# Attach binding fitness values
landscape.attach(name="binding_fitness", values=fintess_binding, dtype="numeric")

# Compute tmap results
# landscape.view("binding_fitness")
# tmap_res_binding = fl.analysis.diffusion_scale.compute_ruggedness_diffusion_scale(landscape, t_min=1e-10, t_max=1e2)

# Attach folding fitness values
landscape.attach(name="folding_fitness", values=fitness_folding, dtype="numeric")
# Compute tmap results
landscape.view("folding_fitness")
tmap_res_folding = fl.analysis.diffusion_scale.compute_ruggedness_diffusion_scale(landscape, t_min=1e-10, t_max=1e2)


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

In [26]:
df_intersect

,sequence,fitness_bind,fitness_fold
0,TLMVALYDYEARTEDDLSFHKGEKFQILNSSEGDWWEARSLTTGET...,0.190655,-0.229669
1,TLMVALYDYEARTEDDLSFHKGEKLQILNSSEGDWWEARSLTTGET...,0.134442,0.010316
2,TLIVALYDYEARTEDDLSFHKGEKLQVLNSSEGDWWEARSLTTGET...,0.067504,-0.806056
3,TLVVALYDYEARTEDDLSFHKGEKLQVLNSSEGDWWEARSLTTGET...,0.067474,-0.955170
4,TLVVALYDYEARTEDDLSLHKGEKFQVLNSSEGDWWEARSLTTGET...,0.042600,-1.062226
...,...,...,...
8569,TLLVALYDYEARTEDDFSFHKGEKIQVLNSSEGDWWEARSLTTGET...,-2.798605,-3.284098
8570,TLMVALYDYEARTEDDFSIHKGEKLQVLNSSEGDWWEARSLTTGET...,-2.829492,-2.990955
8571,TLIVALYDYEARTEDDMSMHKGEKVQLLNSSEGDWWEARSLTTGET...,-2.885729,-3.592901
8572,TLIVALYDYEARTEDDFSVHKGEKMQFLNSSEGDWWEARSLTTGET...,-2.997841,-3.697020
